# AI SkyRun · Tier-A — навчи дрон летіти за вже готовою траєкторією

Траєкторія між деревами **вже задана** — той самий оптимальний шлях, який шукає
A* оракул у базовому кіті. Тут дрон не планує маршрут: він отримує **6 ступенів
свободи** (`x,y,z, φ,θ,ψ`, тобто roll/pitch/yaw) і мусить навчитися крутити торком
(torque) навколо трьох осей, щоб утриматись на цій траєкторії — під випадковими
поривами вітру.

Дрон **не бачить дерев і не бачить своєї абсолютної позиції**. Він бачить лише
**помилку відносно плану**: наскільки він збочив убік (`e_lat`), по висоті
(`e_alt`), і наскільки його крен/тангаж/курс відрізняються від того, що велить
траєкторія.

> **Правило гри те саме, що й у Tier 1:** доступу до `env.trees` чи `env.xyz`
> немає. Тільки вектор помилок.

Дві дірки `# TODO` нижче: ціль DQN (Bellman target) і функція втрат (loss).


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from tier_a.env_attitude.env import AttitudeEnv
from tier_a.env_attitude.constants import N_TREES_ATTITUDE
from tier_d.scoring.seeds import PUBLIC_SEEDS

SEED, N_TREES = 0, N_TREES_ATTITUDE
env = AttitudeEnv(n_trees=N_TREES, seed=SEED)

print(f"обсерв.: {env.obs_dim}D   дій: {env.n_actions}")
print(f"довжина референсної траєкторії S* = {env.reference.S_total:.3f} м,  T* = {env.reference.T_total:.3f} с")


## Що дає середовище

| | |
|---|---|
| `obs = env.reset()` | 10 чисел: `e_lat, e_alt, e_vz, e_φ, e_θ, e_ψ, p, q, r, κ` |
| `obs2, r, done, info = env.step(a)` | `a ∈ 0..26` — 27 комбінацій торків (roll/pitch/yaw ∈ {−Δ,0,+Δ}) |
| винагорода | `−1` крок · `−100` колізія/відхилення (кінець) · `+100` ціль (кінець) |
| `info["collision"]`, `info["departed"]`, `info["loss_of_control"]`, `info["goal"]` | справжні термінали |
| `info["truncated"]` | скінчився час польоту — **це не термінал** |
| `info["wind"]` | цього кроку стався порив вітру (торк збурено) |

Тяга зафіксована на висоті ховера (`T = m·g`) — це **не дія**. Керуєте
лише трьома торками. Це свідомо звужує задачу до утримання орієнтації, а не
повної 3D-оптимізації траєкторії.


## TODO 1 — ціль DQN (Bellman target) і функція втрат

$$\text{target} = \begin{cases} r & \text{термінал} \\ r + \gamma \max_{a'} Q_{\text{target}}(s', a') & \text{інакше} \end{cases}$$

$$\mathcal{L} = \big(Q(s,a) - \text{target}\big)^2$$

`target_net` — заморожена копія мережі, яка синхронізується періодично.
Рахувати ціль із **тієї самої** мережі, яку зараз оновлюєте — класична причина
переоцінених Q-значень (перевірено емпірично: без цього жадібна політика була
**гіршою** за випадкову).


In [ ]:
def dqn_update(qnet, target_net, optimizer, batch, gamma):
    """Один крок градієнтного спуску на мінібатчі. Повертає loss (для дебагу)."""
    s, a, r, s2, terminal = batch    # batch-и тензорів: terminal -- bool-тензор (B,)
    q_sa = qnet(s).gather(1, a.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        # TODO(1): та сама логіка, що в tier_d — термінал → r; інакше → r + γ·max Q_target(s2).
        #   (~terminal).float() = 0 у терміналі (гасить майбутнє), 1 інакше.
        target = r + gamma * target_net(s2).max(dim=1).values * (~terminal).float()

    # TODO(2): loss тягне Q(s,a) до target — квадратична помилка (як у формулі)
    loss = ((q_sa - target) ** 2).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return float(loss.item())

## Reward shaping (дано, не дірка)

Той самий потенціал-орієнтований прийом, що й у Tier 1, тільки Φ тепер —
зважена норма помилки відстеження, а не відстань до цілі:

$$\Phi(s) = -\big(w_{pos}\sqrt{e_{lat}^2+e_{alt}^2} + w_{att}\sqrt{e_\phi^2+e_\theta^2+e_\psi^2}\big)$$

І, як і раніше: **Φ(термінал) = 0** — інакше гарантія «shaping не змінює
оптимум» не діє.


In [ ]:
W_POS, W_ATT = 1.0, 0.3

def potential(obs):
    """Φ(s) = -(зважена норма помилки відстеження). Тільки власна obs."""
    e_lat, e_alt = obs[0], obs[1]
    e_phi, e_theta, e_psi = obs[3], obs[4], obs[5]
    return -(W_POS * math.hypot(e_lat, e_alt) + W_ATT * math.hypot(e_phi, e_theta, e_psi))

def shaping_F(obs, obs2, gamma, terminal=False):
    phi_next = 0.0 if terminal else potential(obs2)
    return gamma * phi_next - potential(obs)


class QNet(nn.Module):
    def __init__(self, obs_dim=10, n_actions=27, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )
    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    def __init__(self, capacity=20000, seed=0):
        from collections import deque
        import random
        self.buf = deque(maxlen=capacity)
        self._rng = random.Random(seed)
    def push(self, s, a, r, s2, terminal):
        self.buf.append((s, a, r, s2, terminal))
    def sample(self, batch_size):
        batch = self._rng.sample(self.buf, min(batch_size, len(self.buf)))
        s, a, r, s2, terminal = zip(*batch)
        return (torch.as_tensor(np.array(s), dtype=torch.float32),
                torch.as_tensor(a, dtype=torch.long),
                torch.as_tensor(r, dtype=torch.float32),
                torch.as_tensor(np.array(s2), dtype=torch.float32),
                torch.as_tensor(terminal, dtype=torch.bool))
    def __len__(self):
        return len(self.buf)


## Гіперпараметри

`gamma` дисконт · `lr` темп навчання мережі · `eps` дослідження (ε-greedy) ·
`train_every` — робити крок градієнту не на кожному кроці середовища
(інакше навчання занадто повільне), а раз на кілька кроків.

> `EPISODES = 150` — це **смок-конфіг**: секунди, не хвилини, щоб швидко
> перевірити, що код не зламаний. Це важка 6-DOF задача з нуля — для
> справжньої збіжності підніміть `EPISODES` до 1200+ (хвилини).


In [ ]:
GAMMA = 0.99
LR = 1e-3
EPS0, EPS_MIN = 1.0, 0.05
EPISODES = 150     # смок-конфіг: секунди, не хвилини. Підніміть для повної збіжності.
TRAIN_EVERY = 4
TARGET_SYNC_EVERY = 200
BATCH_SIZE = 64


In [ ]:
def greedy_rollout_local(env, qnet, max_steps=2000):
    """argmax_a Q(s,a), без дослідження. Так вас оцінюватимуть."""
    obs = env.reset()
    sq = []
    for _ in range(max_steps):
        with torch.no_grad():
            a = int(qnet(torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)).argmax(dim=1).item())
        obs2, r, done, info = env.step(a)
        sq.append(obs2[0]**2 + obs2[1]**2)
        obs = obs2
        if done or info["truncated"]:
            break
    rmse = float(np.sqrt(np.mean(sq))) if sq else 0.0
    return {"success": bool(info["goal"]), "collision": bool(info["collision"]),
            "departed": bool(info["departed"] or info["loss_of_control"]),
            "tracking_rmse": rmse, "steps": len(sq)}


def train_local(env, episodes=EPISODES, seed=0):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    qnet = QNet(env.obs_dim, env.n_actions)
    target_net = QNet(env.obs_dim, env.n_actions)
    target_net.load_state_dict(qnet.state_dict())
    optimizer = torch.optim.Adam(qnet.parameters(), lr=LR)
    buffer = ReplayBuffer(seed=seed)
    decay = max(1, int(episodes * 0.6))
    curve = []
    global_step, update_count = 0, 0

    for ep in range(episodes):
        eps = max(EPS_MIN, EPS0 + (EPS_MIN - EPS0) * ep / decay)
        obs = env.reset()
        while True:
            if rng.random() < eps:
                a = int(rng.integers(env.n_actions))
            else:
                with torch.no_grad():
                    a = int(qnet(torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)).argmax(dim=1).item())
            obs2, r, done, info = env.step(a)
            global_step += 1
            terminal = info["collision"] or info["goal"] or info["departed"] or info["loss_of_control"]
            r_train = r + shaping_F(obs, obs2, GAMMA, terminal)
            buffer.push(obs, a, r_train, obs2, terminal)

            if len(buffer) >= BATCH_SIZE and global_step % TRAIN_EVERY == 0:
                dqn_update(qnet, target_net, optimizer, buffer.sample(BATCH_SIZE), GAMMA)
                update_count += 1
                if update_count % TARGET_SYNC_EVERY == 0:
                    target_net.load_state_dict(qnet.state_dict())

            obs = obs2
            if done or info["truncated"]:
                break

        if ep % 25 == 0 or ep == episodes - 1:
            roll = greedy_rollout_local(env, qnet)
            curve.append((ep, roll["tracking_rmse"]))
    return qnet, np.array(curve)


qnet, curve = train_local(AttitudeEnv(n_trees=N_TREES, seed=SEED), seed=SEED)
roll = greedy_rollout_local(AttitudeEnv(n_trees=N_TREES, seed=SEED), qnet)
print(f"дійшов: {roll['success']}   tracking_rmse = {roll['tracking_rmse']:.3f}   кроків = {roll['steps']}")


## Крива навчання (tracking RMSE жадібної політики)

150 епізодів — це лише смок-тест. Якщо лінія не спадає чітко вниз —
це очікувано; цей 6-DOF-контроль значно складніший, ніж табличний Q-learning з
Tier 1. Підніміть `EPISODES`, щоб побачити справжню збіжність.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
if len(curve):
    ax.plot(curve[:, 0], curve[:, 1], color="#0969da", lw=1.8, label="жадібна політика")
ax.set_xlabel("епізод"); ax.set_ylabel("tracking RMSE (м)"); ax.legend(); ax.grid(alpha=.3)
plt.show()


## 3D-політ з гізмо крену/тангажу/курсу

In [ ]:
from tier_d.viz.forest3d import plot_flight_3d, draw_attitude_gizmo

def rollout_with_pose(env, qnet, max_steps=2000):
    obs = env.reset()
    xyzs, atts = [env.xyz.copy()], [env.attitude]
    for _ in range(max_steps):
        with torch.no_grad():
            a = int(qnet(torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)).argmax(dim=1).item())
        obs, r, done, info = env.step(a)
        xyzs.append(env.xyz.copy()); atts.append(env.attitude)
        if done or info["truncated"]:
            break
    return np.array(xyzs), atts

e = AttitudeEnv(n_trees=N_TREES, seed=SEED)
xyzs, atts = rollout_with_pose(e, qnet)

fig = plt.figure(figsize=(7, 5.5))
ax = fig.add_subplot(111, projection="3d")
plot_flight_3d(ax, e.trees, xyzs[:, :2], title="Tier-A: жадібна політика")
draw_attitude_gizmo(ax, xyzs[-1], *atts[-1])
plt.show()


## Самоперевірка на public seeds

Фінал рахується на **прихованих** seed-ах, яких ви не бачите.


In [ ]:
print(f"{'seed':>5} {'T*':>7} {'дійшов':>7} {'rmse':>7}")
for s in PUBLIC_SEEDS:
    e = AttitudeEnv(n_trees=N_TREES, seed=s)
    qs, _ = train_local(e, seed=s)
    r = greedy_rollout_local(AttitudeEnv(n_trees=N_TREES, seed=s), qs)
    print(f"{s:>5} {e.reference.T_total:7.3f} {str(r['success']):>7} {r['tracking_rmse']:7.3f}")
